# Volume 12, Chapter 13: Curves over finite fields (Python companion)

Exercises first, solutions at the end. We compute zeta functions from point counts and watch the functional equation and the Riemann hypothesis appear, redo Gauss's count of $x^3+y^3=1$, test the Sato–Tate law, check the twist identity that gave the lower bound in Theorem 13.13, and finish with a classic of computational number theory: **counting the points of an elliptic curve over a 50-bit prime field in a fraction of a second**, by a baby-step giant-step search inside the Hasse interval. Requires `cypari2`.

In [ ]:
import math, random, time
from fractions import Fraction as Fr
import cypari2
pari = cypari2.Pari()
pari.allocatemem(10**9, silent=True)
random.seed(13)

## Exercises

**P13.1 (★) Zeta functions from point counts.** Write a brute-force counter of the projective points of a plane curve $F(X,Y,Z)=0$ over $\mathbb F_{q^r}$ (use PARI's `ffgen` and `ffprimroot` to list the field). From $N_1,\dots,N_g$ compute $P(T)$ (Theorem 13.8: $\log P=\sum_r(N_r-q^r-1)T^r/r$ up to degree $g$, then the functional equation). Do it for $y^2=x^3-x+1$ over $\mathbb F_5$ and for the Klein quartic $X^3Y+Y^3Z+Z^3X$ over $\mathbb F_2$; check $|\alpha_i|=\sqrt q$, compute $h=P(1)$, and check the predicted $N_r$ against brute force for larger $r$.

**P13.2 (★) Gauss's count.** For the primes $p\equiv1\pmod 3$ below $3000$, count the points of $X^3+Y^3=Z^3$ over $\mathbb F_p$ (count cube roots with a table), find $L\equiv1\pmod3$ with $4p=L^2+27M^2$, and check Proposition 13.9: $N=p+1+L$.

**P13.3 (★★) Sato–Tate.** For $E:y^2=x^3-x+1$ and the primes $p<10^6$, compute $x_p=a_p/2\sqrt p$ with PARI's `ellap`. Compare the moments $\mathbb E[x_p^{2k}]$ for $k=1,2,3$ with those of the semicircle law, $C_k/4^k$ ($C_k$ the Catalan numbers). Then do the same for $y^2=x^3-x$, which has complex multiplication by $i$: what fraction of the $a_p$ vanish, and what distribution do the others follow?

**P13.4 (★★) Twists and the Riemann hypothesis.** For $C:y^3=f(x)$ with $f$ a quartic without repeated roots (a smooth plane quartic, $g=3$), count $N=|C(\mathbb F_p)|$ for $p\equiv1\pmod3$ and check the twist identity of Proposition 13.12: $N(C_1)+N(C_c)+N(C_{c^2})=3(p+1)$ for $c$ a non-cube. Then, for 200 random $f$ and primes $p<3000$, find the largest value of $|N-p-1|/(2g\sqrt p)$: does the Hasse–Weil bound ever come close to being attained?

**P13.5 (★★★) CS application: point counting in the Hasse interval.** To use an elliptic curve in cryptography one needs $N=|E(\mathbb F_p)|$ (it must have a large prime factor). Counting by summing Legendre symbols costs $p$ steps. Hasse's theorem says $N\in[p+1-2\sqrt p,\,p+1+2\sqrt p]$, an interval of length $4\sqrt p$, and $N\cdot P=O$ for every point $P$. Implement a **baby-step giant-step** search for the multiples $m$ of the order of a random point $P$ in that interval, in $O(p^{1/4})$ group operations; when several $m$ survive, intersect over several points, and if needed use the quadratic twist $E'$, whose order is $2p+2-N$ (Mestre's trick). Count a random curve over a prime $p\approx10^{15}$ and compare with PARI's `ellcard`.

## Solutions

### P13.1
**How to find it.** A projective point has a unique representative $(x:y:1)$, $(x:1:0)$ or $(1:0:0)$. With $s_r=q^r+1-N_r$, $\log P(T)=-\sum s_rT^r/r$, and the functional equation $c_{2g-j}=q^{g-j}c_j$ gives the top half of the coefficients.

In [ ]:
def field(q):
    # list of the elements of F_q as PARI objects
    p = next(d for d in range(2, q + 1) if q % d == 0)
    if q == p:
        return [pari.Mod(i, p) for i in range(p)]
    g = pari("ffprimroot(ffgen(%d, 'a))" % q)
    return [0 * g] + [g**i for i in range(q - 1)]

def count(F, q):
    K = field(q); one = K[1]**0; zero = 0 * one
    n = sum(1 for x in K for y in K if F(x, y, one) == 0)
    n += sum(1 for x in K if F(x, one, zero) == 0)
    return n + (1 if F(one, zero, zero) == 0 else 0)

def zeta_P(Ns, q, g):
    # coefficients c_0..c_{2g} of P from N_1..N_g
    logP = [Fr(0)] + [Fr(Ns[r - 1] - q**r - 1, r) for r in range(1, g + 1)]
    c = [Fr(1)] + [Fr(0)] * g          # exp of a power series, degree <= g
    for n in range(1, g + 1):
        c[n] = sum(k * logP[k] * c[n - k] for k in range(1, n + 1)) / n
    c = [int(x) for x in c]
    return c + [q**(g - j) * c[j] for j in range(g - 1, -1, -1)]

def predicted(c, q, R):
    roots = [complex(z) for z in pari(f"polroots(Pol({c[::-1]}))")]
    alphas = [1 / z for z in roots]
    return [round((q**r + 1 - sum(a**r for a in alphas)).real) for r in range(1, R + 1)], alphas

E = lambda X, Y, Z: Y**2 * Z - X**3 + X * Z**2 - Z**3
NE = [count(E, 5**r) for r in (1, 2, 3)]
cE = zeta_P(NE[:1], 5, 1)
print("y^2=x^3-x+1 over F_5: N_r =", NE, " P =", cE, " predicted", predicted(cE, 5, 3)[0])

K4 = lambda X, Y, Z: X**3 * Y + Y**3 * Z + Z**3 * X
NK = [count(K4, 2**r) for r in range(1, 7)]
cK = zeta_P(NK[:3], 2, 3)
pred, alphas = predicted(cK, 2, 6)
print("Klein quartic over F_2: brute force", NK)
print("P(T) coefficients", cK, " predicted N_r", pred)
print("|alpha_i| =", sorted(round(abs(a), 12) for a in alphas), " sqrt 2 =", round(math.sqrt(2), 12))
print("h = P(1) =", sum(cK))

For the elliptic curve, $P(T)=1+2T+5T^2$, so $h=P(1)=8=N_1$, as it must be (the classes of degree $0$ are the points). For the Klein quartic, $P(T)=1+5T^3+8T^6$, all six $|\alpha_i|$ equal $\sqrt2$ to twelve digits, $h=14$ (the Jacobian has $14$ rational points), and $N_4,N_5,N_6$ predicted from $N_1,N_2,N_3$ agree with the brute-force counts. Three numbers determine infinitely many.

### P13.2
**How to find it.** $N_{\rm aff}=\sum_a c(a)\,c(1-a)$ with $c(a)=\#\{x:x^3=a\}$, plus $3$ points at infinity. $L$ is found by trying the $L\equiv1\pmod3$ with $|L|<2\sqrt p$.

In [ ]:
def fermat_cubic(p):
    c = [0] * p
    for x in range(p): c[x**3 % p] += 1
    return sum(c[a] * c[(1 - a) % p] for a in range(p)) + (3 if p % 3 == 1 else 1)

def gauss_L(p):
    for L in range(-2 * math.isqrt(p) - 1, 2 * math.isqrt(p) + 2):
        r = 4 * p - L * L
        if L % 3 == 1 and r > 0 and r % 27 == 0 and math.isqrt(r // 27)**2 == r // 27:
            return L

ps = [int(p) for p in pari.primes([7, 3000]) if int(p) % 3 == 1]
ok = all(fermat_cubic(p) == p + 1 + gauss_L(p) for p in ps)
print(len(ps), "primes, Gauss's formula holds for all:", ok)
print([(p, gauss_L(p), fermat_cubic(p)) for p in ps[:6]])

### P13.3
**How to find it.** The semicircle density $\frac2\pi\sqrt{1-x^2}$ on $[-1,1]$ has moments $\int x^{2k}\frac2\pi\sqrt{1-x^2}\,dx=C_k/4^k$: $\frac14,\frac18,\frac5{64}$.

In [ ]:
def xs(a, b, N):
    E = pari.ellinit([a, b]); D = int(pari.ellglobalred(E)[0])
    return [int(pari.ellap(E, p)) / (2 * math.sqrt(p)) for p in pari.primes([3, N]) if D % int(p)]

t = time.time()
X = xs(-1, 1, 10**6)
print(len(X), "primes in %.1fs" % (time.time() - t))
for k, Ck in ((1, 1), (2, 2), (3, 5)):
    print(f"E[x^{2*k}] = {sum(x**(2*k) for x in X)/len(X):.5f}   semicircle {Ck/4**k:.5f}")

Y = xs(-1, 0, 10**6)
zero = sum(1 for x in Y if x == 0) / len(Y)
nz = [x for x in Y if x != 0]
print(f"y^2=x^3-x: fraction of a_p = 0: {zero:.4f}")
# for CM curves, theta_p is uniform on [0, pi] for the non-zero ones: E[cos^2] = 1/2, E[cos^4] = 3/8
print("non-zero ones: E[x^2] = %.4f (uniform angle: 0.5),  E[x^4] = %.4f (0.375)"
      % (sum(x**2 for x in nz) / len(nz), sum(x**4 for x in nz) / len(nz)))

The non-CM curve matches the semicircle moments to three decimals. For $y^2=x^3-x$, $a_p=0$ exactly for the primes $p\equiv3\pmod4$ (half of them, the *supersingular* primes), and for $p\equiv1\pmod4$, $a_p=\pm2a$ where $p=a^2+b^2$ (a Jacobi sum, as in Gauss's last diary entry): the angle $\theta_p$ is then the argument of a Gaussian prime, and these are uniformly distributed (Hecke). Different symmetry, different law.

### P13.4
**How to find it.** $c(a)=\#\{y:y^3=a\}$ by a table; one point at infinity since $\deg f=m+1=4$. A non-cube is a primitive root $g$.

In [ ]:
def count_y3(f, p):
    c = [0] * p
    for y in range(p): c[y**3 % p] += 1
    return sum(c[f(x) % p] for x in range(p)) + 1

def squarefree(coeffs, p):
    P = pari.Pol([pari.Mod(a, p) for a in coeffs])
    return pari.poldegree(P) == 4 and pari.poldegree(pari.gcd(P, pari.deriv(P))) == 0

p = 997; coeffs = [1, 0, -3, 5, 2]    # f = x^4 - 3x^2 + 5x + 2
assert squarefree(coeffs, p)
f = lambda x: sum(a * x**(4 - i) for i, a in enumerate(coeffs))
g = int(pari.znprimroot(p).lift())
Ns = [count_y3(lambda x, c=c: c * f(x), p) for c in (1, g, g * g)]
print("p =", p, "twists:", Ns, " sum =", sum(Ns), " 3(p+1) =", 3 * (p + 1))

worst = 0
for _ in range(200):
    p = random.choice([int(q) for q in pari.primes([100, 3000]) if int(q) % 3 == 1])
    while True:
        cf = [random.randrange(1, p)] + [random.randrange(p) for _ in range(4)]
        if squarefree(cf, p): break
    N = count_y3(lambda x: sum(a * x**(4 - i) for i, a in enumerate(cf)), p)
    worst = max(worst, abs(N - p - 1) / (6 * math.sqrt(p)))
print("largest |N-p-1|/(2g sqrt p) over 200 random quartic curves:", round(worst, 3))

The twist identity holds exactly. The bound $2g\sqrt p$ is never violated, and random curves stay well inside it: $N-p-1=-\sum\alpha_i$ is a sum of six numbers of absolute value $\sqrt p$ with "random" angles, and such a sum rarely has all its terms aligned. Curves that attain the bound (maximal curves, Exercise 13.6) are special.

### P13.5
**How to find it.** Write $m=\mathrm{lo}+is+j$ with $0\le j<s$, $s\approx\sqrt{4\sqrt p}$. Store the baby steps $jP$ by their $x$-coordinate; then $mP=O$ iff $(\mathrm{lo}+is)P=-jP$, which is looked up for $i=0,1,\dots$. Each point gives the set of $m$ in the interval with $mP=O$; their intersection over a few points is $\{N\}$ unless the group has a small exponent, which the twist resolves.

In [ ]:
def ec_add(P, Q, a, p):
    if P is None: return Q
    if Q is None: return P
    (x1, y1), (x2, y2) = P, Q
    if x1 == x2 and (y1 + y2) % p == 0: return None
    if P == Q: lam = (3 * x1 * x1 + a) * pow(2 * y1, -1, p) % p
    else: lam = (y2 - y1) * pow(x2 - x1, -1, p) % p
    x3 = (lam * lam - x1 - x2) % p
    return (x3, (lam * (x1 - x3) - y1) % p)

def ec_mul(k, P, a, p):
    R = None
    while k:
        if k & 1: R = ec_add(R, P, a, p)
        P = ec_add(P, P, a, p); k >>= 1
    return R

def sqrt_mod(n, p):
    r = pari.Mod(n, p)
    return int(pari.sqrt(r).lift()) if pari.issquare(r) else None

def random_point(a, b, p):
    while True:
        x = random.randrange(p); y = sqrt_mod((x**3 + a * x + b) % p, p)
        if y is not None and y: return (x, y)

def bsgs_candidates(a, b, p):
    P = random_point(a, b, p)
    lo = p + 1 - 2 * math.isqrt(p) - 2; hi = p + 1 + 2 * math.isqrt(p) + 2
    s = math.isqrt(hi - lo) + 1
    baby, Q = {}, None
    for j in range(s):
        if Q is not None: baby.setdefault(Q[0], []).append((j, Q[1]))
        Q = ec_add(Q, P, a, p)
    G = ec_mul(s, P, a, p); R = ec_mul(lo, P, a, p); out = set()
    for i in range(s + 1):
        base = lo + i * s
        if R is None: out.add(base)                       # j = 0
        else:
            for j, y in baby.get(R[0], []):               # R = -jP  <=>  same x, opposite y
                if (y + R[1]) % p == 0: out.add(base + j)
        R = ec_add(R, G, a, p)
    return {m for m in out if lo <= m <= hi}

def count_points(a, b, p, tries=6):
    cands = None
    for _ in range(tries):
        S = bsgs_candidates(a, b, p)
        cands = S if cands is None else cands & S
        if len(cands) == 1: return cands.pop()
    # Mestre: the twist d*y^2 = x^3 + a x + b, i.e. Y^2 = X^3 + a d^2 X + b d^3, has order 2p+2-N
    d = next(t for t in range(2, p) if not pari.issquare(pari.Mod(t, p)))
    twist = None
    for _ in range(tries):
        S = {2 * p + 2 - m for m in bsgs_candidates(a * d * d % p, b * d**3 % p, p)}
        twist = S if twist is None else twist & S
    both = cands & twist
    assert len(both) == 1
    return both.pop()

p = int(pari.nextprime(10**15 + random.randrange(10**14)))
a, b = random.randrange(p), random.randrange(p)
t = time.time(); N = count_points(a, b, p); t1 = time.time() - t
t = time.time(); Np = int(pari.ellcard(pari.ellinit([a, b], p))); t2 = time.time() - t
print("p =", p)
print("BSGS:", N, "in %.2fs;  PARI ellcard:" % t1, Np, "in %.3fs" % t2, "  agree:", N == Np)
print("a_p =", p + 1 - N, " |a_p|/2sqrt(p) = %.3f" % (abs(p + 1 - N) / (2 * math.sqrt(p))))
print("factorisation of N:", pari.factor(N))

In [ ]:
# y^2 = x^3 - x has full 2-torsion, so its group is never cyclic; the search still finds N
for p in [int(pari.nextprime(10**9 + k)) for k in (0, 8, 100)]:
    N = count_points(p - 1, 0, p)
    print(p, N, N == int(pari.ellcard(pari.ellinit([-1, 0], p))), pari.ellgroup(pari.ellinit([-1, 0], p)))

A few times $10^4$ group operations replace $10^{15}$ Legendre symbols. PARI is faster still: it uses the *Schoof–Elkies–Atkin* algorithm, which computes $a_p$ modulo many small primes $\ell$ from the action of the Frobenius on the $\ell$-torsion points, in time polynomial in $\log p$, and reassembles it by the Chinese remainder theorem; this is how curves over 256-bit fields such as Curve25519 (Chapter 12) are counted. Both methods rest on Hasse's theorem: BSGS searches the Hasse interval, and SEA needs $|a_p|\le2\sqrt p$ to know that $a_p$ is determined by its residue modulo $\prod\ell>4\sqrt p$.